In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

# sub_grade is added to the columns used in notebook 02
cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
        'home_ownership', 'annual_inc', 'purpose', 'dti', 'loan_status']
df = df[cols]
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f"Total: {len(df):,}")

Total: 1,345,310


In [3]:
# Feature 1: missing-value indicator for emp_length.
# It must be created BEFORE filling the NaNs, otherwise the information is lost.
df['emp_length_missing'] = df['emp_length'].isnull().astype(int)

# Convert emp_length to a number and fill the remaining NaNs with the median (as before)
emp_map = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
           '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}
df['emp_length'] = df['emp_length'].map(emp_map)
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

# Feature 2: loan amount relative to annual income
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

# Feature 3: sub_grade (35 levels) instead of grade (7), encoded as an ordered number
subgrades = sorted(df['sub_grade'].dropna().unique())
subgrade_map = {sg: i+1 for i, sg in enumerate(subgrades)}
df['sub_grade_num'] = df['sub_grade'].map(subgrade_map)

df[['emp_length_missing', 'loan_to_income', 'sub_grade', 'sub_grade_num']].head()

,emp_length_missing,loan_to_income,sub_grade,sub_grade_num
0,0,0.065455,C4,14
1,0,0.380000,C1,11
2,0,0.317460,B4,9
4,0,0.099585,F1,26
5,0,0.351471,C3,13


In [4]:
# Improved feature set: sub_grade_num replaces grade, plus the two new features
improved_features = ['loan_amnt', 'term', 'int_rate', 'sub_grade_num', 'emp_length',
                       'emp_length_missing', 'loan_to_income', 'home_ownership',
                       'annual_inc', 'purpose', 'dti']

X = df[improved_features].copy()
y = df['default']

X['term'] = X['term'].str.extract(r'(\d+)').astype(float)
X = pd.get_dummies(X, columns=['home_ownership', 'purpose'], drop_first=True)
X['dti'] = X['dti'].fillna(X['dti'].median())

print(f"Final shape: {X.shape}")
X.isnull().sum().sum()  # should be 0

Final shape: (1345310, 27)


np.int64(0)

In [5]:
# loan_to_income divides by annual_inc, so loans with annual_inc = 0 produce
# infinite values, which XGBoost rejects
print("Infinite values in X:", np.isinf(X.select_dtypes(include=[np.number])).sum().sum())
print("Loans with annual_inc = 0:", (df['annual_inc'] == 0).sum())

Infinite values in X: 361
Loans with annual_inc = 0: 361


In [6]:
# Replace the infinities with a high but finite value (2x the largest valid ratio):
# a loan with no declared income is a very high ratio, just not literally infinite
max_valid = X.loc[~np.isinf(X['loan_to_income']), 'loan_to_income'].max()
X['loan_to_income'] = X['loan_to_income'].replace([np.inf, -np.inf], max_valid * 2)

# Check that no infinities remain
print("Infinite values after the fix:", np.isinf(X.select_dtypes(include=[np.number])).sum().sum())

Infinite values after the fix: 0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Same hyperparameters as the original model in notebook 02, so the comparison
# isolates the effect of the new features
improved_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
improved_model.fit(X_train, y_train)

proba_pred = improved_model.predict_proba(X_test)[:, 1]
auc_improved = roc_auc_score(y_test, proba_pred)

# 0.7120 is the AUC of the original XGBoost model from notebook 02
print("AUC-ROC original model (grade + base features): 0.7120")
print(f"AUC-ROC improved model (sub_grade + new features): {auc_improved:.4f}")
print(f"Improvement: {(auc_improved - 0.7120)*100:.2f} points")

AUC-ROC original model (grade + base features): 0.7120
AUC-ROC improved model (sub_grade + new features): 0.7147
Improvement: 0.27 points


In [8]:
# Feature importance of the improved model
feature_importance_v2 = pd.Series(improved_model.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance_v2.head(10)

sub_grade_num              0.504380
term                       0.095275
home_ownership_MORTGAGE    0.086248
home_ownership_RENT        0.061689
emp_length_missing         0.059172
loan_to_income             0.042860
dti                        0.026767
purpose_small_business     0.019836
int_rate                   0.016243
purpose_wedding            0.010536
dtype: float32